In [28]:
# importing libraries

import re 
import pandas as pd
import numpy as np

In [17]:
# load the file here
# Set the path of the foalder, because of there are four sheets.
data_path = "D:/D/Projects/Healthcare Analytics/data/raw"

# Load excel files
admissions_file = pd.read_csv(data_path+"/admissions.csv")
claims_file = pd.read_csv(data_path+"/claims.csv")
patients_file = pd.read_csv(data_path+"/patients.csv")
readmissions_file = pd.read_csv(data_path+"/readmissions.csv")

In [18]:
# finding shape of each tables
# Why this matters professionally:

# Row counts confirm:
# dataset generated correctly
# no file truncation happened
# joins later won’t silently fail
table_name = ['admission_file', 'claims_file', 'patients_file', 'readmissions_file']
tables = [admissions_file, claims_file, patients_file, readmissions_file]

def tables_shape():
    for name, file in zip(table_name, tables):
        shape = file.shape
        print(name + ': ', shape)

tables_shape()

admission_file:  (200000, 11)
claims_file:  (200000, 9)
patients_file:  (10200, 10)
readmissions_file:  (3000, 8)


In [19]:
# Clumn overview
# Finding information of sheets which tells missing value, data types, column presence, and memory footprint
def tables_info():
    for name, file in zip(table_name, tables):
        print("Information for the " + name)
        info = file.info()

tables_info()

Information for the admission_file
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   admission_id         200000 non-null  object
 1   patient_id           200000 non-null  object
 2   admit_date           200000 non-null  object
 3   discharge_date       200000 non-null  object
 4   department           194000 non-null  object
 5   diagnosis_code       199488 non-null  object
 6   diagnosis_desc       200000 non-null  object
 7   severity             200000 non-null  object
 8   attending_physician  200000 non-null  object
 9   los_days             200000 non-null  int64 
 10  outcome              200000 non-null  object
dtypes: int64(1), object(10)
memory usage: 16.8+ MB
Information for the claims_file
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
 #   Colum

In [20]:
# CHECKING PRIMARY KEYS
# In relational dataset, primary key is essential part of the data table. It must always be unique. 

primary_key = [admissions_file['admission_id'].is_unique, 
               claims_file['claim_id'].is_unique,
               patients_file['patient_id'].is_unique,
               readmissions_file['readmit_id'].is_unique]

for result in primary_key:
    print(result)

True
True
True
True


In [21]:
# DUPLICATION DETECTION
# 1. Admissions file
patients_file.duplicated(subset=['full_name', 'dob']).sum()

np.int64(200)

In [22]:
# NULL VALUE SCAN
# This identifies:which columns are damaged and how badly they are damaged
def tables_null_value():
    for name, file in zip(table_name, tables):
        print("Information for ", name)
        result = file.isnull().sum()
        print(result)
        print(" *************************************** ")
tables_null_value()

Information for  admission_file
admission_id              0
patient_id                0
admit_date                0
discharge_date            0
department             6000
diagnosis_code          512
diagnosis_desc            0
severity                  0
attending_physician       0
los_days                  0
outcome                   0
dtype: int64
 *************************************** 
Information for  claims_file
claim_id                0
admission_id            0
billed_amount           0
approved_amount         0
payer                   0
claim_status         4000
denial_reason      159963
submission_date         0
payment_date        99789
dtype: int64
 *************************************** 
Information for  patients_file
patient_id        0
full_name         0
dob               0
gender            0
blood_type        0
insurance_type    0
region            0
signup_date       0
phone             0
email             0
dtype: int64
 *************************************** 
I

In [23]:
# Finding perc of null values in table
def tables_null_percentage():
    for name, file in zip(table_name, tables):
        print("PERCENTAGE -",name)
        result = (file.isnull().mean()*100).round(2)
        print(result)
        print("")
tables_null_percentage()

PERCENTAGE - admission_file
admission_id           0.00
patient_id             0.00
admit_date             0.00
discharge_date         0.00
department             3.00
diagnosis_code         0.26
diagnosis_desc         0.00
severity               0.00
attending_physician    0.00
los_days               0.00
outcome                0.00
dtype: float64

PERCENTAGE - claims_file
claim_id            0.00
admission_id        0.00
billed_amount       0.00
approved_amount     0.00
payer               0.00
claim_status        2.00
denial_reason      79.98
submission_date     0.00
payment_date       49.89
dtype: float64

PERCENTAGE - patients_file
patient_id        0.0
full_name         0.0
dob               0.0
gender            0.0
blood_type        0.0
insurance_type    0.0
region            0.0
signup_date       0.0
phone             0.0
email             0.0
dtype: float64

PERCENTAGE - readmissions_file
readmit_id               0.0
patient_id               0.0
original_admission_id    0.0
r

In [24]:
# DATE LOGIC VALIDATION
# This counts invalid hospital stays. These are high-risk errors in healthcare analytics.
def tables_date_logic_validation():
    admissions_file['admit_date'] = pd.to_datetime(admissions_file['admit_date'])
    admissions_file['discharge_date'] = pd.to_datetime(admissions_file['discharge_date'])
    result = (admissions_file['discharge_date'] < admissions_file['admit_date']).sum()
    print(result)
tables_date_logic_validation()


2000


In [25]:
# NEGATIVE VALUE DETACTION
# Negative LOS = impossible medically
# Negative billing = impossible financially
# These must be flagged.
print((admissions_file['los_days'] < 0).sum())
print((claims_file['billed_amount'] < 0).sum())


2000
2000


In [26]:
# REFERENTIAL INTEGRITY
# referentail integrity is a relationa dataset concept, which insures that relationship between table_name remain constent and accurate.
refrential_intg = [(~admissions_file['patient_id'].isin(patients_file['patient_id'])).sum(),
                   (~claims_file['admission_id'].isin(admissions_file['admission_id'])).sum(),
                   (~readmissions_file['patient_id'].isin(patients_file['patient_id'])).sum(),
                   (~readmissions_file['original_admission_id'].isin(admissions_file['admission_id'])).sum()
                   ]
refrential_intg

[np.int64(1000), np.int64(0), np.int64(19), np.int64(0)]

INSPECT DIAGNOSIS CLUMNS


In [30]:
# Inspect Diagnosis column first
# Main reason behind this we will confirm that column is exist, see raw patterns, and catch weird spacing and formating issues
admissions_file[['diagnosis_code', 'diagnosis_desc']].head(10)

,diagnosis_code,diagnosis_desc
0,E11.9,Type 2 diabetes without complications
1,N18.3,"Chronic kidney disease, stage 3"
2,S72.001,Femur fracture
3,J44.1,COPD with acute exacerbation
4,G35,Multiple sclerosis
5,S72.001,Femur fracture
6,I63.9,"Cerebral infarction, unspecified"
7,K92.1,Melena
8,E11.9,Type 2 diabetes without complications
9,I63.9,"Cerebral infarction, unspecified"


In [37]:
# count missing diagnosis values in percentages
round(admissions_file[['diagnosis_code', 'diagnosis_desc']].isna().mean() * 100, 2)

diagnosis_code    0.26
diagnosis_desc    0.00
dtype: float64

In [ ]:
# Perform standardization in diagnosis column, because diagnosis_code contains space, unwanted chaacter, and formatting issues
admissions_file["diagnosis_code_clean"] = (
        admissions_file['diagnosis_code']
        .astype("string")
        .str.strip()
        .str.upper
)


<bound method StringMethods.upper of <pandas.core.strings.accessor.StringMethods object at 0x00000252C94D9CF0>>

In [42]:
# Appply ICD style formate validation
icd_pattern = r"^[A-Z][0-9]{2}(\.[A-Z0-9]{1,3})?$"
admissions_file['icd_format_valid'] = admissions_file['diagnosis_code_clean'].str.match(
    icd_pattern,
    na=False
)

In [45]:
invalid_icd_count = (~admissions_file["icd_format_valid"]).sum()
invalid_icd_pct = round((~admissions_file["icd_format_valid"]).mean() * 100, 2)

print("Invalid ICD format count:", invalid_icd_count)
print("Invalid ICD format %:", invalid_icd_pct)

Invalid ICD format count: 200000
Invalid ICD format %: 100.0


In [49]:
# Inspect valid example
admissions_file.loc[
    ~admissions_file['icd_format_valid'],['diagnosis_code', 'diagnosis_code_clean', 'diagnosis_desc']
].drop_duplicates().head(5)

,diagnosis_code,diagnosis_code_clean,diagnosis_desc
0,E11.9,<bound method StringMethods.upper of <pandas.c...,Type 2 diabetes without complications
1,N18.3,<bound method StringMethods.upper of <pandas.c...,"Chronic kidney disease, stage 3"
2,S72.001,<bound method StringMethods.upper of <pandas.c...,Femur fracture
3,J44.1,<bound method StringMethods.upper of <pandas.c...,COPD with acute exacerbation
4,G35,<bound method StringMethods.upper of <pandas.c...,Multiple sclerosis


In [52]:
# Sperater missing and malformed data
admissions_file['diagnosis_code_missing_flag'] = admissions_file['diagnosis_code_clean'].isna()
admissions_file['diagnosis_code_invalid_format_flag'] = (
    ~admissions_file['diagnosis_code_missing_flag'] & ~admissions_file['icd_format_valid']
)

print("Missing diagnosis code: ", admissions_file['diagnosis_code_missing_flag'].sum())
print("Malformed diagnosis codes: ", admissions_file['diagnosis_code_invalid_format_flag'].sum())

Missing diagnosis code:  0
Malformed diagnosis codes:  200000
